In [ ]:
from PIL import Image
import numpy as np
import pandas as pd

img = Image.open("frames/frame_00003.png").convert("RGBA")

In [ ]:
width, height = img.size

data = np.array(img)  # shape: (H, W, 4)

In [ ]:
gray = (
    0.299 * data[:, :, 0] +
    0.587 * data[:, :, 1] +
    0.114 * data[:, :, 2]
)

In [ ]:
gray = gray.astype(int)

In [ ]:
print(gray.shape)

In [ ]:
print(gray)

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(gray, cmap="gray")
plt.title("Grayscale Frame")
plt.show()

In [ ]:
region = gray[60:70, 150:170]
pd.DataFrame(region)

In [ ]:
region = gray[100:110, 50:70]
pd.DataFrame(region)

In [ ]:
region = gray[120:130, 180:190]
pd.DataFrame(region)

In [ ]:

mask = np.where(gray == 255, 255, 0).astype(np.uint8)
plt.imshow(mask, cmap="gray")

In [ ]:
region = mask[50:100, 18:20]
pd.DataFrame(region)

In [ ]:
from scipy.ndimage import generate_binary_structure, label, binary_closing
import numpy as np
#mask = binary_closing(mask, structure=np.ones((3,3)))
structure = generate_binary_structure(2,2)  # 8-connectivity
labeled, n = label(mask, structure=structure)
def detect_obstacles(gray):

      # 8-connectivity
    objects = []

    for i in range(1, n + 1):
        ys, xs = np.where(labeled == i)
        
        size = np.sum(labeled == i)
        print(f"Object {i}: {size} pixels")
        
        if len(xs) < 100:
            continue
        
        x_min = int(np.min(xs))
        x_max = int(np.max(xs))
        y_min = int(np.min(ys))
        y_max = int(np.max(ys))
        
        objects.append({
            #"cx": float(np.mean(xs)),
            #"cy": float(np.mean(ys)),
            "size": len(xs),
            "x_min": x_min,
            "x_max": x_max,
            "y_min": y_min,
            "y_max": y_max
        })

    if not objects:
        return None

    nearest = min(objects, key=lambda o: o["x_min"])

    return objects, nearest

In [ ]:
obj, nearest = detect_obstacles(gray)
print(obj)
print(nearest)

In [ ]:
radius = int(np.sqrt(4))  # rough estimate
cx, cy = int(45.5), int(98.5)

x1 = max(cx - radius, 0)
x2 = cx + radius + 1

y1 = max(cy - radius, 0)
y2 = cy + radius + 1

region = mask[y1:y2, x1:x2]
print(x1, x2, y1, y2)